# 02 — Demand Forecasting

In [1]:
%cd /content
!test -d demand-forecasting-inventory-optimization || git clone https://github.com/hemanthkarthikd/demand-forecasting-inventory-optimization.git
%cd /content/demand-forecasting-inventory-optimization/notebooks

/content
Cloning into 'demand-forecasting-inventory-optimization'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 21 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 39.45 KiB | 1.64 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/demand-forecasting-inventory-optimization/notebooks


In [2]:
import pandas as pd
import numpy as np
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
demand = demand.sort_values(["sku_id","week_start"])

In [3]:
def mape_nonzero(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100

rows = []
forecast_rows = []

for sku_id, g in demand.groupby("sku_id"):
    g = g.sort_values("week_start").reset_index(drop=True)
    train, test = g.iloc[:-13].copy(), g.iloc[-13:].copy()
    y_train = train["demand_units"].astype(float)
    y_test = test["demand_units"].astype(float)

    naive = np.repeat(y_train.iloc[-1], len(test))
    ma4 = np.repeat(y_train.tail(4).mean(), len(test))

    try:
        model = ExponentialSmoothing(
            y_train, trend="add", seasonal="add",
            seasonal_periods=52, initialization_method="estimated"
        ).fit(optimized=True)
        hw = np.clip(model.forecast(len(test)).values, 0, None)
    except Exception:
        hw = ma4.copy()

    methods = {"Naive":naive, "Moving Average (4W)":ma4, "Holt-Winters":hw}
    for method, pred in methods.items():
        rows.append([
            sku_id, method,
            mean_absolute_error(y_test,pred),
            sqrt(mean_squared_error(y_test,pred)),
            mape_nonzero(y_test,pred)
        ])
        for dt,a,f in zip(test["week_start"],y_test,pred):
            forecast_rows.append([sku_id,dt,method,a,float(f)])

metrics = pd.DataFrame(rows, columns=["sku_id","method","mae","rmse","mape_pct"])
forecasts = pd.DataFrame(forecast_rows, columns=["sku_id","week_start","method","actual_demand","forecast_demand"])
metrics.to_csv("../data/processed/forecast_metrics.csv", index=False)
forecasts.to_csv("../data/processed/forecast_holdout.csv", index=False)

best = metrics.loc[metrics.groupby("sku_id")["rmse"].idxmin()].sort_values("sku_id")
best.to_csv("../data/processed/best_forecast_method_by_sku.csv", index=False)

display(metrics.groupby("method")[["mae","rmse","mape_pct"]].mean().sort_values("rmse"))
display(best)

,mae,rmse,mape_pct
method,,,
Holt-Winters,41.055679,51.844599,17.412769
Moving Average (4W),43.149840,52.746470,18.277388
Naive,50.506410,59.112753,20.091569


,sku_id,method,mae,rmse,mape_pct
0,SKU-001,Naive,9.846154,12.440628,8.008361
5,SKU-002,Holt-Winters,59.340386,83.040310,13.685559
8,SKU-003,Holt-Winters,41.277505,56.918355,10.603758
9,SKU-004,Naive,26.923077,32.893768,12.734825
13,SKU-005,Moving Average (4W),14.519231,18.023089,10.238578
16,SKU-006,Moving Average (4W),135.807692,147.868770,40.687966
18,SKU-007,Naive,34.153846,37.015589,19.567953
23,SKU-008,Holt-Winters,30.775278,41.235880,10.283127
26,SKU-009,Holt-Winters,32.211523,41.415479,11.615192
28,SKU-010,Moving Average (4W),29.923077,37.364113,11.276189
